# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [19]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
#assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'
#print(os.environ.get('OPENAI_API_KEY'))

client = AsyncOpenAI(
    api_key='voc-185726272021403752018696a901dce4924e3.99464742',
    base_url='https://openai.vocareum.com/v1'
    )

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [20]:
DATA_DIR = Path('./data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [ ]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot."""
    return [
        {
            "role": "user",
            "content": f"""Extract the following information from the job description:
- company
- role
- years_experience_required

If no required years of experience are stated, use null.
Return the result as JSON.

Job description:
{snippet_text}
"""
        }
    ]

def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot."""
    return [
        {
            "role": "user",
            "content": f"""Extract company, role, and required years of experience from job descriptions.
Below are some examples:

Example 1:
Snippet: "Acme Corp is hiring a Senior Software Engineer. Candidates should have at least 5 years of experience."
Answer:
{{
    "company": "Acme Corp",
    "role": "Senior Software Engineer",
    "years_experience_required": 5
}}

Example 2:
Snippet: "Northwind Ltd is looking for a Data Analyst with a minimum of 2 years of experience."
Answer:
{{
    "company": "Northwind Ltd",
    "role": "Data Analyst",
    "years_experience_required": 2
}}

Example 3:
Snippet: "Cyberdyne Systems is hiring an AI/ML Research Scientist. No experience requirement is specified."
Answer:
{{
    "company": "Cyberdyne Systems",
    "role": "AI/ML Research Scientist",
    "years_experience_required": null
}}

Now extract the information from this job description:

{snippet_text}

Return ONLY JSON.
"""
        }
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based."""
    return [
        {
            "role": "system",
            "content": """You are an expert recruiter.

Analyse the job description and extract the following information:

1. company
2. role
3. years_experience_required

Return ONLY valid JSON with these exact fields:

{
    "company": "company name",
    "role": "job role",
    "years_experience_required": 0
}

If the job description does not state a required number of years,
return null for years_experience_required.

Do not add any other fields.
"""
        },
        {
            "role": "user",
            "content": f"""Extract the required information from this job description:
{snippet_text}"""
        }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought."""
    return [
        {
            "role": "user",
            "content": f"""Analyse the following job description carefully.
Think through the information provided before giving your final answer.

Extract:
- company
- role
- years_experience_required

If no required years of experience are stated, use null.
Return ONLY valid JSON using exactly this format:

{{
    "company": "company name",
    "role": "job role",
    "years_experience_required": 0
}}

Job description:
{snippet_text}"""
        }
    ]


STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [22]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response."""
    
    text = text.strip()

    # Remove markdown code fences
    if text.startswith("```"):
        lines = text.splitlines()

        # Remove first line such as ```json
        if lines:
            lines = lines[1:]

        # Remove closing ```
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    # Try to parse the complete response
    try:
        parsed = json.loads(text)

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass

    # If there is extra text around the JSON, try extracting {...}
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        try:
            parsed = json.loads(text[start:end + 1])

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            pass

    return None

async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet."""

    prompt_function = STRATEGIES[strategy_name]

    messages = prompt_function(snippet["snippet"])

    start_time = time.perf_counter()

    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE
    )

    latency = time.perf_counter() - start_time

    raw_response = response.choices[0].message.content or ""

    extracted = parse_response(raw_response)

    # Token usage
    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens

    # Calculate cost
    cost_usd = (
        prompt_tokens * RATES[MODEL]["in"]
        + completion_tokens * RATES[MODEL]["out"]
    )

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw_response": raw_response,
        "extracted": extracted,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost_usd": cost_usd,
        "latency_s": latency,
    }

async def run_all() -> list[dict]:
    """Run all 10 x 4 = 40 calls in parallel."""

    tasks = []

    for snippet in snippets:
        for strategy_name in STRATEGIES:
            tasks.append(
                run_one(strategy_name, snippet)
            )

    results = await asyncio.gather(*tasks)

    return results

In [23]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': 'Question: What are the qualifications required for the Senior Software Engineer position at Acme Corp?\n\nAnswer: The qualifications required for the Senior Software Engineer position at Acme Corp include 5+ years of backend development experience and strong skills in Python and distributed systems.',
 'extracted': None,
 'prompt_tokens': 62,
 'completion_tokens': 51,
 'cost_usd': 3.9899999999999994e-05,
 'latency_s': 2.824330700095743}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare company, role, and years_experience_required."""

    if extracted is None:
        return 0

    score = 0

    # Company
    extracted_company = extracted.get("company")
    gold_company = gold.get("company")

    if (
        isinstance(extracted_company, str)
        and isinstance(gold_company, str)
        and extracted_company.strip().lower() == gold_company.strip().lower()
    ):
        score += 1

    # Role
    extracted_role = extracted.get("role")
    gold_role = gold.get("role")

    if (
        isinstance(extracted_role, str)
        and isinstance(gold_role, str)
        and extracted_role.strip().lower() == gold_role.strip().lower()
    ):
        score += 1

    # Years of experience
    extracted_years = extracted.get("years_experience_required")

    if extracted_years == gold.get("years_experience_required"):
        score += 1

    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4."""

    judge_prompt = f"""You are evaluating an LLM extracting information from a job description.
Original job description:{snippet_text}

GOLD ANSWER:{json.dumps(gold, indent=2)}

MODEL ANSWER:{json.dumps(extracted, indent=2) if extracted is not None else "UNPARSABLE"}

Evaluate these three fields:
1. company
2. role
3. years_experience_required

Rubric:

4 — all three fields are correct.
3 — two of three fields are correct, with no fabricated information.
2 — one of three fields is correct, OR the model fabricated information.
1 — none are correct OR the response is unparsable.

For years_experience_required:
- null is correct when no experience requirement is stated.
- Do not penalize reasonable interpretation of phrases such as "7+ years" → 7.
- For a range such as "3-5 years", the minimum (3) is correct.
- "fresh graduates welcome" means 0 years.
- Spelled-out numbers such as "three years" should be interpreted as 3.

Return ONLY valid JSON:

{{
    "score": 1
}}

The score must be an integer from 1 to 4.
"""

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": judge_prompt
            }
        ],
        temperature=0.0
    )

    text = response.choices[0].message.content or ""
    parsed = parse_response(text)

    if parsed is None:
        return 1

    try:
        score = int(parsed.get("score"))

        if 1 <= score <= 4:
            return score

    except (TypeError, ValueError):
        pass

    return 1

In [16]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added
print(f'Scored {len(scored)} results.')

Scored 0 results.


## Step 5 — Build the comparison table

In [18]:
df = pd.DataFrame(scored)

summary = df.groupby("strategy").agg({
    "accuracy": "mean",
    "parse_success": "mean",
    "llm_judge_score": "mean",
    "cost": "sum",
    "latency": "median",
}).round(3)

summary.columns = [
    "Accuracy (mean)",
    "Parse rate",
    "Judge score",
    "Cost ($)",
    "Latency p50 (s)"
]

summary

KeyError: 'strategy'

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```